<a href="https://colab.research.google.com/github/gomesf126/Projeto-An-lise-de-Vendas-com-Python-Pandas-/blob/main/ProjetoAnaliseDeVendas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [49]:
import pandas as pd
import os

caminho = "/content/drive/MyDrive/Colab Notebooks/Vendas"
lista_arquivo = os.listdir(caminho)
tabelas=[]#para o meu dataframe

for arquivo in lista_arquivo:
  nome = arquivo.lower()
  if "vendas" in nome and "devolucoes" not in nome:
    caminho_arquivo = os.path.join(caminho,arquivo) #juntar o caminho com o nome do arquivo
    #tratamento de erros
    try:
      #padronizar o pré-fixo/acessa o DataFrame do arquivo
      tabela = pd.read_csv(caminho_arquivo, sep=',' ,encoding='utf-8')
      #limpar colunas
      tabela.columns= tabela.columns.str.strip().str.lower()
      #remover colunas lixo
      tabela = tabela.loc[:, ~tabela.columns.str.contains('^unnamed', case=False)]
      #garantir colunas
      colunas_necessarias = ['produto', 'quantidade vendida']
      for col in colunas_necessarias:
        if col not in tabela.columns:
          tabela[col] = None

      tabelas.append(tabela)
    except Exception as e:
      print(f"Erro ao processar o arquivo: {arquivo}: {e}")


if tabelas:
  df_tabelas = pd.concat(tabelas, ignore_index=True)
else:
  df_tabelas = pd.DataFrame()

  print("Nenhuma tabela encontrada! .")


#display(df_tabelas.isna().sum())
#display(df_tabelas.info())
display(df_tabelas.head(20))

display(df_tabelas.columns.tolist())
#TRATAMENTO E PADRONIZAÇÃO
#NOME
df_tabelas['primeiro nome'] =(df_tabelas['primeiro nome']
                              .fillna('Não Informado')
                              .str.title()
                              .str.strip())
df_tabelas['data'] = df_tabelas['data'].ffill()

#PREÇO

df_tabelas['preco unitario'] = (
    df_tabelas['preco unitario']
    .astype(str)
    .str.replace(r'[^\d,]', '', regex=True)  # 🔥 remove R$, espaço etc
    .str.replace('.', '', regex=False) #Remover milhar
    .str.replace(',', '.', regex=False)#ajusta para decimal
)
#converte em número do tipo float e o que não conseguir deixa nulo
df_tabelas['preco unitario'] = pd.to_numeric(df_tabelas['preco unitario'], errors='coerce')
#trata o campo nulo
df_tabelas['preco unitario'] = df_tabelas['preco unitario'].fillna(df_tabelas['preco unitario'].mean())
#quantidade vendida

df_tabelas['quantidade vendida'] = df_tabelas['quantidade vendida'].astype(str).str.replace(r'[^\d]', '', regex=True)#só aceita número
df_tabelas['quantidade vendida'] = pd.to_numeric(df_tabelas['quantidade vendida'], errors='coerce')#converte para numero caso não seja número, Valores inválidos → NaN
df_tabelas['quantidade vendida'] =  df_tabelas['quantidade vendida'].fillna(0).astype(int)#numero do formato inteiro

#DATA
df_tabelas['data'] = pd.to_datetime(
    df_tabelas['data'].astype(str).str.strip(),
    errors='coerce'
)

display(df_tabelas.info())



# FEATURE ENGINEERING
df_tabelas['faturamento'] =(df_tabelas['quantidade vendida'] * df_tabelas['preco unitario']).round(2)
#MÊS
df_tabelas['mes'] = df_tabelas['data'].dt.month
#ANALISE
#Faturamento mensal
top_mes = (df_tabelas.groupby('mes' , as_index=False)
          .agg(Faturamento = ('faturamento', 'sum'))
          .sort_values(by='Faturamento', ascending=False)
           )#
#qual o ticket médio

#agrupar os produtos que vendem em quantidade
top_produto =(df_tabelas.groupby('produto', as_index=False)
             .agg(quantidade =('quantidade vendida', 'sum'))
             )#
#qual o produto que mais vendido na loja
mais_vende = top_produto.loc[top_produto['quantidade'].idxmax()]
#qual o produto que menos vende na loja
menos_vende =top_produto.loc[top_produto['quantidade'].idxmin()]

print(f"produto que mais vende: {mais_vende['produto']} quantidade {mais_vende['quantidade']}")
print(f"produto que menos vende: {menos_vende['produto']} quantidade {menos_vende['quantidade']}")

#qual o produto que vende em pequena quantidade e gera um bom faturamento
#baixa quantidade + alto faturamento
baixo_qt_alto_fat  = (df_tabelas.groupby('produto' , as_index=False)
                   .agg(em_quantidade = ('quantidade vendida', 'sum'),
                        faturamento =('faturamento','sum'))
                   .sort_values(by=['em_quantidade' ,'faturamento'], ascending=[True, False])
                   )#
display(baixo_qt_alto_fat )
#qual o produto que vende grande quantidade e gera um bom faturamento
#alta quantidade + alto faturametno
maior_qt_fat =(df_tabelas.groupby('produto', as_index=False)
              .agg(em_quantidade =('quantidade vendida','sum'),
                   faturamento =('faturamento','sum'))
              .sort_values(by=['em_quantidade','faturamento'], ascending=[False,False])
               )#

display(maior_qt_fat)
#qual a loja que mais fatura
#qual a loja que menos fatura


#qual o produto mais barato
#qual o produto mais caro
produto_preco=(df_tabelas.groupby('produto', as_index=False)
              .agg(preco_max=('preco unitario','max'),
                   preco_min=('preco unitario','min'),
                   preco_medio=('preco unitario','mean'))
              )#este modelo evita do erro de pegar o valor promocional ou algum erro de digitação
mais_caro = produto_preco.sort_values(by='preco_medio', ascending=False).head(5)
mais_barato = produto_preco.sort_values(by='preco_medio', ascending=True).head(5)
#“Eu evito usar max/min direto porque pode ter outliers. Prefiro média ou mediana para representar melhor o comportamento real.”


#Produto caro que vende pouco / premium
#Produto barato que vende muito /muito volume
#Produto caro que vende muito / gera maior lucro na empresa
display(mais_caro)
display(mais_barato)
#qual o faturamento de dois produtos nos ultimos 6 meses com gráfico/dashboard





,sku,produto,quantidade vendida,primeiro nome,sobrenome,data,loja,preco unitario
0,HL4379,Televisão,4,Jessica,Cordovil,2/11/2018,Rio de Janeiro,2500
1,HL1918,iPhone,2,Camilla,Guimarães,10/4/2018,Rio de Janeiro,5300
2,HL4379,Televisão,5,Katharina,Barros,3/26/2018,Rio de Janeiro,2500
3,HL9962,Android,1,Rodrigo,Silva,1/14/2018,Rio de Janeiro,3400
4,HL4379,Televisão,3,Júlio,Freire,6/30/2018,Rio de Janeiro,2500
5,HL1148,Câmera,5,Jackson,Derossi,11/30/2018,Rio de Janeiro,2100
6,HL1148,Câmera,1,Willian,Albino,1/30/2018,Rio de Janeiro,2100
7,HL2714,Tablet,4,Jéssica,Resinente,2/26/2018,Rio de Janeiro,1600
8,HL1918,iPhone,3,Stephanie,Gonçalves,9/24/2018,Rio de Janeiro,5300
9,HL8851,Notebook,1,Guilherme,Vianna,4/24/2018,Rio de Janeiro,3500


['sku',
 'produto',
 'quantidade vendida',
 'primeiro nome',
 'sobrenome',
 'data',
 'loja',
 'preco unitario']

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9999 entries, 0 to 9998
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   sku                 9999 non-null   object        
 1   produto             9999 non-null   object        
 2   quantidade vendida  9999 non-null   int64         
 3   primeiro nome       9999 non-null   object        
 4   sobrenome           9999 non-null   object        
 5   data                9999 non-null   datetime64[ns]
 6   loja                9999 non-null   object        
 7   preco unitario      9999 non-null   int64         
dtypes: datetime64[ns](1), int64(2), object(5)
memory usage: 625.1+ KB


None

produto que mais vende: iPhone quantidade 8974
produto que menos vende: Câmera quantidade 2805


,produto,em_quantidade,faturamento
1,Câmera,2805,5890500
4,Tablet,2921,4673600
3,SmartWatch,2980,4172000
0,Android,3183,10822200
2,Notebook,3249,11371500
5,Televisão,5931,14827500
6,iPhone,8974,47562200


,produto,em_quantidade,faturamento
6,iPhone,8974,47562200
5,Televisão,5931,14827500
2,Notebook,3249,11371500
0,Android,3183,10822200
3,SmartWatch,2980,4172000
4,Tablet,2921,4673600
1,Câmera,2805,5890500


,produto,preco_max,preco_min,preco_medio
6,iPhone,5300,5300,5300.0
2,Notebook,3500,3500,3500.0
0,Android,3400,3400,3400.0
5,Televisão,2500,2500,2500.0
1,Câmera,2100,2100,2100.0


,produto,preco_max,preco_min,preco_medio
3,SmartWatch,1400,1400,1400.0
4,Tablet,1600,1600,1600.0
1,Câmera,2100,2100,2100.0
5,Televisão,2500,2500,2500.0
0,Android,3400,3400,3400.0
